# 第4周 · 线性回归与梯度下降（含数学工作坊 1）

**本周结束时你能做到**：用 X、y、模型、损失四要素描述一个监督学习问题；从零实现梯度下降并解释学习率的作用；说明为什么要做特征标准化；用正规方程一步解出线性回归，并说明 XᵀX 不可逆时 Ridge 为什么仍有唯一解。

本周是课程的分水岭：前三周在学“怎么让计算机听话”，从本周起学“怎么让计算机**从数据中学习**”。学习的核心机制——梯度下降——你将亲手实现，而且它和第 20 周训练大模型用的是同一个机制。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from zero2researcher import data
from zero2researcher.checks import week04 as ck

---
## S1 · 数学工作坊 1：向量、矩阵与“导数即斜率”

不用紧张：今天的每一个数学概念都立刻用 NumPy 验证。它们其实都是第 2 周用过的东西，今天只是给它们正式的名字。

### 向量与点积

向量 = 一列数 = 一维 NumPy 数组。**点积**：对应位置相乘、再求和。

先用循环写一个手工版 `my_dot(a, b)`，再和 NumPy 对比：

In [ ]:
def my_dot(a, b):
    """点积：对应位置相乘再求和"""
    # 在这里写你的代码
    raise NotImplementedError

a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
print(my_dot(a, b), np.dot(a, b), a @ b)    # 三种写法，同一个数
ck.dot(my_dot)

点积马上就有用：**线性模型的一次预测就是一次点积**——每个特征乘一个权重，再加起来。

### 矩阵乘法与 shape 规则

矩阵 = 数字表格 = 二维数组。**本课程最重要的约定：每一行是一个样本，每一列是一个特征。** `X.shape == (n, d)` 读作“n 个样本、d 个特征”。

矩阵乘法 `A @ B`：结果的第 i 行第 j 列 = A 的第 i 行与 B 的第 j 列的点积。shape 规则（内侧维度相等、然后被“消掉”）：

```
(n, d) @ (d, 1) → (n, 1)        # 矩阵 @ 矩阵
(n, d) @ (d,)   → (n,)          # 矩阵 @ 向量（我们最常用）
```

In [ ]:
X = np.array([[1.0, 2.0],
              [3.0, 4.0],
              [5.0, 6.0]])      # (3, 2)：3 个样本、2 个特征
w = np.array([10.0, 1.0])       # (2,)：每个特征一个权重

print((X @ w).shape, X @ w)     # (3,)：每个样本一个预测值
print(X[0] @ w)                 # 第一个样本的预测 = 一次点积

内侧维度对不上就会报错——这是接下来十六周你最常见的错误，现在亲眼看一次（这个单元格**应该**报错）：

In [ ]:
w_bad = np.array([1.0, 2.0, 3.0])    # (3,)：X 的内侧是 2，对不上
X @ w_bad

最后一行的 `mismatch` 就是“内侧维度对不上”。**万能排查法：把每个数组的 `.shape` 打印出来，对着规则核对。**

### 导数就是斜率

机器学习的“学习”由一个问题驱动：**参数往哪个方向动一点，损失会变小？**答案由导数给出。导数是函数在一点的斜率——放大到足够近，任何光滑曲线看起来都是直线。

不用背求导公式，可以用**数值差分**直接把斜率“量”出来：在 x 两侧各取一小步，割线的斜率就近似切线的斜率。写 `num_derivative(f, x, h=1e-5)`，返回 `(f(x+h) - f(x-h)) / (2h)`：

In [ ]:
def num_derivative(f, x, h=1e-5):
    """数值差分近似 f 在 x 处的导数（斜率）"""
    # 在这里写你的代码
    raise NotImplementedError

def square(x):
    return x ** 2

print(num_derivative(square, 3.0))    # 教科书答案：2x = 6
print(num_derivative(np.sin, 0.0))    # 教科书答案：cos(0) = 1
ck.numerical_derivative(num_derivative)

亲眼看一下“导数即斜率”：

In [ ]:
xs = np.linspace(-3, 3, 100)
x0 = 1.5
slope = num_derivative(square, x0)

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(xs, square(xs), label="f(x) = x^2")
ax.plot(xs, square(x0) + slope * (xs - x0), "--", label=f"tangent at x={x0} (slope {slope:.1f})")
ax.scatter([x0], [square(x0)], color="k", zorder=3)
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.set_title("The derivative is the slope of the tangent line")
ax.legend()
plt.show()

斜率为正 → 往左走函数值变小；斜率为负 → 往右走变小。**往斜率的反方向走，函数值就下降。**记住这句话——一小时后它就是梯度下降的全部。

---
## S2 · 监督学习的四要素

**机器学习**：不再手写规则，而是给出数据和“什么算好”的标准，让算法自己找规则。监督学习用四样东西描述：

| 要素 | 含义 | 今天的例子 |
| --- | --- | --- |
| 数据 X | 特征，shape (n, d) | 一个输入变量 x |
| 标签 y | 想预测的量，shape (n,) | 连续数值（所以叫“回归”） |
| 模型 | 从 X 到预测值的函数族 | 直线 y_hat = w·x + b |
| 损失 | 一个数，衡量预测差多远 | 均方误差 MSE |

学习 = 找到让损失最小的参数 (w, b)。

### 合成数据：自带标准答案的“标准品”

研究新方法时，先在**自己造的、答案已知的数据**上验证——这就是你实验室里的阳性对照。按 y = 3x + 2 + 噪声 造 200 个点（**写死随机种子**：可复现实验的第一步）：

In [ ]:
rng = np.random.default_rng(42)          # 随机种子固定为 42
X = rng.uniform(-2, 2, size=(200, 1))    # (n, d) = (200, 1)：200 个样本、1 个特征
y = 3 * X[:, 0] + 2 + rng.normal(0, 1, size=200)

true_w, true_b = 3.0, 2.0                # 只有造数据的人才知道的“标准答案”

fig, ax = plt.subplots(figsize=(5, 3))
ax.scatter(X[:, 0], y, s=10, alpha=0.6)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Synthetic data: y = 3x + 2 + noise")
plt.show()

### 损失：均方误差（MSE）

预测 y_hat 与真实 y 的差叫**残差**。MSE 就是残差平方的平均值：

```
MSE = mean((y_hat - y) ** 2)
```

平方让正负误差不相互抵消，也让大错误受到更重的惩罚。写出来：

In [ ]:
def mse_loss(y_hat, y):
    """均方误差：残差平方的平均"""
    # 在这里写你的代码
    raise NotImplementedError

print(mse_loss(np.array([1.0, 2.0, 3.0]), np.array([1.0, 1.0, 5.0])))   # (0 + 1 + 4) / 3
ck.mse_loss(mse_loss)

### 损失曲面：每个 (w, b) 对应一个损失

参数只有两个，可以奢侈一把：把候选参数全试一遍，对网格上的每个 (w, b) 算一次 MSE，画等高线图：

In [ ]:
ws = np.linspace(-1, 7, 81)
bs = np.linspace(-2, 6, 81)
loss_grid = np.zeros((len(bs), len(ws)))
for i, b_ in enumerate(bs):
    for j, w_ in enumerate(ws):
        loss_grid[i, j] = mse_loss(w_ * X[:, 0] + b_, y)

fig, ax = plt.subplots(figsize=(5, 4))
cs = ax.contour(ws, bs, loss_grid, levels=20)
ax.clabel(cs, inline=True, fontsize=7)
ax.scatter([true_w], [true_b], marker="*", s=150, color="red", label="true (w, b)")
ax.set_xlabel("w")
ax.set_ylabel("b")
ax.set_title("MSE loss surface: one bowl, bottom near (3, 2)")
ax.legend()
plt.show()

一个碗，碗底就是最优参数。但有个问题：**真实模型有上百万个参数，这张图画不出来**。模型能做的，只是站在当前这一点，感知脚下的坡度——这就够了。

### 手算一步梯度下降

先把 b 固定在 2，只调 w。损失关于 w 的导数（用微积分可以推出来，第 6 周工作坊会推）：

```
dL/dw = mean(2 * (y_hat - y) * x)
```

梯度下降的更新规则，就是刚才那句话：**往斜率的反方向走一小步**——

```
w ← w - 学习率 × 导数
```

In [ ]:
w_now, b_now = 0.0, 2.0
lr = 0.1                                   # 学习率：一步走多远

y_hat = w_now * X[:, 0] + b_now
grad_w = np.mean(2 * (y_hat - y) * X[:, 0])
print("当前 w =", w_now, "  损失 =", round(mse_loss(y_hat, y), 3), "  导数 =", round(grad_w, 3))

w_new = w_now - lr * grad_w
print("更新后 w =", round(w_new, 3), "  损失 =", round(mse_loss(w_new * X[:, 0] + b_now, y), 3))

导数是负的（往右损失会降），所以 w 增大、损失确实下降了。

顺手做一次 sanity check：用 S1 的数值差分验证解析导数没写错（第 7 周的“梯度检查”就是这个思想）：

In [ ]:
print("解析导数:", round(grad_w, 5))
print("数值导数:", round(num_derivative(lambda w_: mse_loss(w_ * X[:, 0] + b_now, y), w_now), 5))

---
## S3 · 从零实现向量化梯度下降

把“一次调一个参数”升级成“全部参数一起更新”。流程固定为五步，**这也是你以后每个训练脚本的骨架**：数据 → 模型 → 损失 → 更新循环 → 看 loss 曲线，最后与标准答案对比。

模型的向量化写法（X 是 (n, d)，w 是 (d,)，对任意多特征都成立）：

```
y_hat = X @ w + b
```

两个参数的梯度（解析式，d 个特征时 grad_w 是 (d,) 向量）：

```
grad_w = 2/n * X.T @ (y_hat - y)
grad_b = 2/n * sum(y_hat - y)
```

> 对照其他教材时注意：d2l、吴恩达的课程把损失写成 $\tfrac12$MSE（吴恩达写作 $\tfrac{1}{2m}\sum$），求导后系数 2 正好被消掉，所以你会看到没有 2 的梯度式。两种写法只差一个常数倍——相当于学习率差 2 倍，学到的最优参数完全一样。

写出一步更新 `gd_step`：

In [ ]:
def gd_step(X, y, w, b, lr):
    """做一步梯度下降，返回更新后的 (w, b)"""
    # 在这里写你的代码
    raise NotImplementedError

ck.gd_step(gd_step)

### 训练循环

从 w=0、b=0 出发，反复调用 `gd_step`，并记录每一步的损失：

In [ ]:
def train(X, y, lr, n_iters):
    w = np.zeros(X.shape[1])
    b = 0.0
    losses = []
    # 在这里写你的代码
    raise NotImplementedError
    return w, b, losses

w_fit, b_fit, losses = train(X, y, lr=0.1, n_iters=200)
print(f"学到的 w = {w_fit[0]:.3f}, b = {b_fit:.3f}（造数据用的是 w=3, b=2）")

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(losses)
ax.set_xlabel("Iteration")
ax.set_ylabel("MSE loss")
ax.set_title("Loss falls fast, then flattens at the noise floor")
plt.show()

loss 曲线是训练的心电图，以后每次训练都先看它。损失降到 1 附近就不再降——因为数据里混了标准差为 1 的噪声，**完美的模型也只能到这里**。学到的参数非常接近标准答案 (3, 2)。

### 学习率：梯度下降唯一要调的旋钮

太小 → 走得太慢；太大 → 一步迈过碗底、越荡越高（发散）。扫一遍亲眼看：

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
for lr in [1e-4, 1e-3, 1e-2, 1e-1, 1.0]:
    _, _, losses_lr = train(X, y, lr=lr, n_iters=100)
    ax.plot(np.minimum(losses_lr, 1e4), label=f"lr={lr:g}")   # 发散的曲线截在 1e4，防止把别人压扁
ax.set_yscale("log")
ax.set_xlabel("Iteration")
ax.set_ylabel("MSE loss (clipped at 1e4)")
ax.set_title("Too small: crawls. Too large: diverges. lr=0.1 fits this problem")
ax.legend()
plt.show()

lr=1.0 的曲线一路向上：每一步都迈过头，离碗底越来越远。lr=1e-4 则在 100 步里几乎没动。**没有万能学习率，只有对当前问题合适的学习率**——作业里你会把它做成一次正式的受控实验。

### 与 sklearn 对比：阳性对照

`sklearn.linear_model.LinearRegression` 用最小二乘公式直接解出最优参数，正好当我们的“标准品”：

In [ ]:
from sklearn.linear_model import LinearRegression

# 在这里写你的代码
raise NotImplementedError
print(f"sklearn:   w = {sk_model.coef_[0]:.6f}, b = {sk_model.intercept_:.6f}")
print(f"我们的 GD: w = {w_fit[0]:.6f}, b = {b_fit:.6f}")
print(f"差距: {abs(sk_model.coef_[0] - w_fit[0]):.2e}, {abs(sk_model.intercept_ - b_fit):.2e}")
ck.fitted_params(w_fit, b_fit)

差距小于亿分之一——**我们从零写的二十行代码，和工业级库给出同一个答案**。

既然 sklearn 一行就能解，为什么还要学梯度下降？因为最小二乘的求解公式**只对线性回归存在**；从第 7 周起的每一个神经网络，都只能靠梯度下降训练。今天的 `gd_step` 循环，就是你未来所有训练代码的祖先。

### sklearn 那一行里是什么：正规方程

线性回归是少数能"一步解出"最优参数的模型，而且不需要新的微积分。碗底就是梯度为 0 的地方：把上面 `gd_step` 里的两个梯度都令为 0——

```
grad_w = 2/n * X.T @ (X @ w + b - y) = 0
grad_b = 2/n * sum(X @ w + b - y)    = 0
```

为了把截距也装进同一个式子，用一个常见技巧（NumPy 文档里 `np.linalg.lstsq` 的示例就是这么写的）：在 X 右边拼一列 1，记作 A = [X, 1]，参数记作 θ = (w, b)，于是 `A @ θ` 正好是 `X @ w + b`。两个条件合起来就是

```
(Aᵀ A) θ = Aᵀ y        # 正规方程（normal equation）
```

这是一个 (d+1)×(d+1) 的线性方程组，`np.linalg.solve` 一步解出。写 `normal_equation(X, y)`，返回 `(w, b)`：

In [ ]:
def normal_equation(X, y):
    """解正规方程 (AᵀA)θ = Aᵀy，其中 A = [X, 1]；返回 (w, b)"""
    # 在这里写你的代码
    raise NotImplementedError

w_ne, b_ne = normal_equation(X, y)
print(f"正规方程:  w = {w_ne[0]:.10f}, b = {b_ne:.10f}")
print(f"sklearn:   w = {sk_model.coef_[0]:.10f}, b = {sk_model.intercept_:.10f}")
print(f"我们的 GD: w = {w_fit[0]:.10f}, b = {b_fit:.10f}")
ck.normal_equation(normal_equation)

三种方法给出同一组参数，差别在小数点后第 15 位（浮点舍入）。

### 为什么不总用它

正规方程一步到位、没有学习率要调，为什么还要梯度下降？三个理由：

1. **只有线性回归（和它的近亲）有这样的闭式解。** 第 5 周的逻辑回归，令梯度为 0 解不出显式公式；神经网络更不用说。它们只能一步步迭代——这就是上面那句"只对线性回归存在"的意思。
2. **代价。** 要先算出 (d+1)×(d+1) 的 AᵀA，再解方程组，计算量大约随 d³ 增长；神经网络动辄几百万个参数，根本解不动。梯度下降每一步只做几次矩阵乘法。
3. **AᵀA 可能不可逆。** 特征之间有完全的线性关系时（例如同一个指标记了两次），方程组没有唯一解。S4 末尾会亲眼看到，并用 Ridge 修好它。

顺便一提：sklearn 的 `LinearRegression` 解的是同一个最小二乘问题，但它不直接解这个方程组，而是调用更稳的最小二乘求解器（`scipy.linalg.lstsq`，基于奇异值分解）。这里两者答案相同；什么时候会不一样，也留到 S4 末尾。

---
## S4 · 真实数据：California Housing

1990 年美国人口普查的加州房价数据：20,640 个街区 × 8 个特征，目标是街区房价中位数（**单位：10 万美元**）。第一次运行会联网下载约 1 MB 并缓存到本机，之后离线可用。

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing(as_frame=True)
X_house, y_house = housing.data, housing.target
print(X_house.shape)
print(list(X_house.columns))
X_house.describe().round(2)

特征包括街区的收入中位数（MedInc，单位万美元）、房龄、平均房间数、人口、经纬度等。先看最有希望的特征——收入 vs 房价：

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.scatter(X_house["MedInc"], y_house, s=2, alpha=0.2)
ax.set_xlabel("Median income (tens of thousands of USD)")
ax.set_ylabel("Median house value (hundreds of thousands of USD)")
ax.set_title("Income predicts house value, but prices are capped at 5.0")
plt.show()

这张图里有两个真实数据的“怪相”，都值得写进实验记录：

- 顶部一条**水平带**：房价在 5.0（50 万美元）被**截断**了——再贵的街区也记成 5.0；
- `describe()` 里 `AveRooms` 最大值 141 间/户？那是住户极少的度假区街区的统计伪影。

**看到怪数据，先查数据的出处和说明，再怀疑自己的代码。**官方说明就在 `housing.DESCR` 里：AveRooms 的怪值它有交代（户数少、空房多的街区，比如度假区，“每户房间数”会大得出奇）；但 **5.0 的截断它一个字也没提**——这只能从数据本身看出来。画一张目标值的直方图：

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(y_house, bins=50)
ax.set_xlabel("Median house value (hundreds of thousands of USD)")
ax.set_ylabel("Number of districts")
ax.set_title("A lone spike at the right edge: the target is capped at 5.0")
plt.show()
print("最大值:", y_house.max(), "  取到最大值的街区数:", int((y_house == y_house.max()).sum()))

最右边那根孤零零的高柱就是截断：近千个街区的房价都被记成了上限 5.00001。**文档不会告诉你一切——先看数据。**

### 单特征回归：只用收入能预测多准？

划分训练/测试集，在训练集上拟合、在**测试集**上算 R²（R² = 1 完美预测，R² = 0 相当于只会猜均值）：

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_house, y_house, test_size=0.25, random_state=42)

# 在这里写你的代码
raise NotImplementedError
print(f"只用 MedInc 的测试 R² = {r2_single:.3f}")
ck.calhousing_r2_single(r2_single)

### 全部 8 个特征

In [ ]:
# 在这里写你的代码
raise NotImplementedError
print(f"全部 8 个特征的测试 R² = {r2_all:.3f}")
ck.calhousing_r2_all(r2_all)

0.47 → 0.59：更多特征确实有帮助，但离 1 还远——房价还取决于数据里根本没有的东西（学区、房屋状况、年代……）。**特征的上限决定模型的上限。**

### 把我们的梯度下降搬上真实数据：会翻车

In [ ]:
X_raw = X_train.to_numpy()
y_raw = y_train.to_numpy()

_, _, losses_diverge = train(X_raw, y_raw, lr=1e-6, n_iters=50)
print(f"lr=1e-6：50 步后损失 = {losses_diverge[-1]:.3g}（发散！）")

_, _, losses_tiny = train(X_raw, y_raw, lr=1e-7, n_iters=200)
print(f"lr=1e-7：200 步后损失 = {losses_tiny[-1]:.3f}（不发散，但离最优的 0.52 还很远）")

症结：8 个特征的尺度天差地别——MedInc 大约 0–15，Population 动辄上千。同一个学习率对有的参数太大（发散）、对有的又太小（不动）。损失曲面不再是圆碗，而是一条又窄又长的峡谷。

### 解药：标准化

每个特征减去自己的均值、除以自己的标准差——正是第 2 周的 z-score！所有特征就站上了同一尺度。**注意：均值和标准差只在训练集上计算**（为什么？第 5 周讲数据泄漏时揭晓）：

In [ ]:
mu = X_raw.mean(axis=0)
sigma = X_raw.std(axis=0)
X_std = (X_raw - mu) / sigma            # 广播：第 2 周学过

w_std, b_std, losses_std = train(X_std, y_raw, lr=0.1, n_iters=200)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(losses_tiny, label="raw features, lr=1e-7")
ax.plot(losses_std, label="standardized, lr=0.1")
ax.set_yscale("log")
ax.set_xlabel("Iteration")
ax.set_ylabel("MSE loss")
ax.set_title("Standardization turns the canyon back into a bowl")
ax.legend()
plt.show()

from sklearn.metrics import r2_score
X_test_std = (X_test.to_numpy() - mu) / sigma     # 测试集用训练集的 mu 和 sigma
print("标准化 + 我们的梯度下降，测试 R² =", round(r2_score(y_test, X_test_std @ w_std + b_std), 3))

标准化之后，我们的梯度下降与 sklearn 并肩（R² ≈ 0.59）。**从第 7 周起，标准化（或它的变体）是喂给神经网络的所有数据的标配。**

### 三方对账：真实数据上

R² 一样，参数也一样吗？在同一份标准化训练集上，把三种解法放在一起：正规方程、sklearn、我们的梯度下降（lr = 0.1、200 步，即前面标准化那一格训练出的 `w_std`），再加一个多走 1800 步的梯度下降：

In [ ]:
import pandas as pd

sk_std = LinearRegression().fit(X_std, y_raw)                        # sklearn，同一份标准化训练集
w_gd2000, b_gd2000, _ = train(X_std, y_raw, lr=0.1, n_iters=2000)    # 梯度下降多走 1800 步
w_ne_h, b_ne_h = normal_equation(X_std, y_raw)                       # 正规方程

table = pd.DataFrame({"normal eq": np.r_[w_ne_h, b_ne_h],
                      "sklearn": np.r_[sk_std.coef_, sk_std.intercept_],
                      "GD 200 steps": np.r_[w_std, b_std],
                      "GD 2000 steps": np.r_[w_gd2000, b_gd2000]},
                     index=list(X_house.columns) + ["b (intercept)"])
print(table.round(4))
print()
print("测试 R²：正规方程", round(r2_score(y_test, X_test_std @ w_ne_h + b_ne_h), 4),
      "  GD 200 步", round(r2_score(y_test, X_test_std @ w_std + b_std), 4))
print(f"与正规方程的最大系数差：sklearn {np.abs(sk_std.coef_ - w_ne_h).max():.1e}   "
      f"GD 200 步 {np.abs(w_std - w_ne_h).max():.3f}   GD 2000 步 {np.abs(w_gd2000 - w_ne_h).max():.1e}")
ck.calhousing_normal_eq(w_ne_h, b_ne_h)

读这张表：

- 正规方程和 sklearn 一致到小数点后十几位——两种算法，同一个答案；
- 测试 R² 两者都是 0.5911，可 **200 步的梯度下降，系数还差 0.06**（Latitude 与 Longitude 两行），走到 2000 步才对上（差约 2e-9）。

为什么 R² 早就到顶，系数却还在挪？纬度和经度高度负相关（相关系数 −0.92：加州的街区大多沿着西北—东南走向的海岸分布，越往北越往西）。把两者的权重**同时**加大同样的量，预测几乎不变，因为标准化后的"纬度 + 经度"在数据里变化很小。所以标准化之后，损失曲面沿这个方向仍是一条又平又长的峡谷：梯度下降沿着它走得很慢（实测要 596 步，系数差才进入 1e-3），但走没走到底，几乎不影响预测。

**R² 对上 ≠ 参数对上。** 如果你关心的是参数本身（例如要解读"哪个特征的系数大"），先确认优化真的收敛了；正规方程在这里就是阳性对照。

### 当 XᵀX 不可逆：同一个指标记了两次

实验数据里常有这种事：同一个指标用两种单位各记了一次（ng/mL 与 µg/mL），或者芯片上同一个基因有两个探针。模拟一下：把标准化后的 MedInc 原样再拼一列。现在 9 列里只有 8 列是"独立"的（矩阵的**秩**是 8），AᵀA 不可逆：把 MedInc 的权重挪一点给它的副本，预测一模一样，所以最小二乘有无穷多个解。看看几种求解方法各自怎么反应（这一格全部给出，直接运行）：

In [ ]:
# 这就是 S3 的正规方程（拼一列 1，再解 (AᵀA)θ = Aᵀy）；展开写，是为了看 solve 本身的反应
X_dup = np.column_stack([X_std, X_std[:, 0]])              # 第 9 列是 MedInc 的一份副本
A_dup = np.column_stack([X_dup, np.ones(len(X_dup))])
A_std = np.column_stack([X_std, np.ones(len(X_std))])
print("X_dup 有", X_dup.shape[1], "列，秩只有", np.linalg.matrix_rank(X_dup))
print(f"AᵀA 的条件数：没有副本时 {np.linalg.cond(A_std.T @ A_std):.0f}，有副本时 {np.linalg.cond(A_dup.T @ A_dup):.1e}")

# ① 手写的正规方程：不同的电脑，反应不同
try:
    theta_solve = np.linalg.solve(A_dup.T @ A_dup, A_dup.T @ y_raw)
    print(f"solve 没有报错，给出 MedInc = {theta_solve[0]:.4f}、副本 = {theta_solve[8]:.4f}"
          f"（两者之和 {theta_solve[0] + theta_solve[8]:.4f}）")
except np.linalg.LinAlgError as err:
    print("solve 报错：", err)

# ② 最小二乘求解器（sklearn 的 LinearRegression 用的也是这一类）：不报错，在无穷多个解里挑一个
theta_ls = np.linalg.lstsq(A_dup, y_raw, rcond=None)[0]
print(f"lstsq：MedInc = {theta_ls[0]:.4f}、副本 = {theta_ls[8]:.4f}（只有一列时是 {sk_std.coef_[0]:.4f}）")

# ③ 无穷多个解：把权重在两列之间挪来挪去，训练 MSE 纹丝不动
for t in [0, 10, 1000]:
    th = theta_ls.copy()
    th[0] += t
    th[8] -= t
    print(f"  MedInc 加 {t:4d}、副本减 {t:4d}：训练 MSE = {mse_loss(A_dup @ th, y_raw):.10f}")

读一读上一格的输出：

- **条件数**从 43 跳到 1e15 以上。条件数衡量"方程组对微小扰动有多敏感"：1e15 意味着数据第 16 位上的一点舍入误差，都可能被放大成解里的显著变化——这个方程组已经没法可靠地求解了。
- **手写的 `solve`**：结果取决于你的电脑用的线性代数库。有的直接报错 `Singular matrix`（奇异矩阵，即不可逆）；有的不报错、给出一个"答案"——课程开发用的 Mac（Apple Accelerate 库）给出 0.387 / 0.465，换一台电脑给的可能完全不同。同一段代码，有的电脑报错，有的电脑给出一个答案，而且每台给的不一样：**两种结果都不可信**。你看到的是哪一种都正常，下面的内容对两种情况都成立。
- **`lstsq`**（以及 sklearn 的 `LinearRegression`）不报错，而是在无穷多个解里挑"权重平方和最小"的那个——两列平分，各 0.426。这是一个合理的约定，但要知道：它是约定，不是数据告诉你的。
- 训练 MSE 在挪动 1000 之后都不变：数据根本区分不了这些解。

研究上的教训：两个完全相同（或高度相关）的特征，**各自的系数没有意义**，只有它们的和有意义（0.852，和只有一列时一样）。读论文里"某基因的系数最大"之类的结论时，先问一句：有没有和它高度相关的特征分走了权重？

### Ridge：加一点 λ，方程组就有唯一解

在损失里加一项惩罚权重大小的项：

```
损失 = Σ (y_hat - y)² + λ · Σ w²
```

（注意这里是平方**和**、不是平均，这是 sklearn `Ridge` 的约定，这样我们的 λ 就和它的参数 `alpha` 完全相等。）同样令梯度为 0，正规方程变成

```
(Xcᵀ Xc + λI) w = Xcᵀ (y - ȳ)，   b = ȳ - x̄ · w
```

其中 Xc = X − x̄ 是**中心化**后的 X（每列减去自己的均值），ȳ 是 y 的均值。为什么要中心化：截距 b 不应该被惩罚（把所有 y 加 100，最优的 b 也应该跟着加 100，而不是被往 0 拉）；先把 X 和 y 都减去均值，截距就从方程里消失了，最后再单独算回来。

为什么这次一定有唯一解：XcᵀXc 的特征值都 ≥ 0（有重复列时，最小的那个恰好是 0，所以不可逆）；加上 λI，每个特征值都增加 λ，全部 ≥ λ > 0——**矩阵一定可逆**。写 `ridge_closed_form(X, y, lam)`：

In [ ]:
def ridge_closed_form(X, y, lam):
    """Ridge 闭式解：先中心化（截距不惩罚），解 (XcᵀXc + λI) w = Xcᵀ(y − ȳ)，返回 (w, b)"""
    # 在这里写你的代码
    raise NotImplementedError

ck.ridge_closed_form(ridge_closed_form)

In [ ]:
from sklearn.linear_model import Ridge

w_r, b_r = ridge_closed_form(X_dup, y_raw, lam=1.0)
sk_r = Ridge(alpha=1.0).fit(X_dup, y_raw)
# 同理，Xc_dup 和下面条件数里的 XcᵀXc + λI 就是 ridge_closed_form 的写法；展开写，是为了看这个矩阵的条件数
Xc_dup = X_dup - X_dup.mean(axis=0)
X_dup_test = np.column_stack([X_test_std, X_test_std[:, 0]])
print(f"λ = 1：MedInc = {w_r[0]:.5f}、副本 = {w_r[8]:.5f}；与 sklearn Ridge(alpha=1) 的最大差 {np.abs(w_r - sk_r.coef_).max():.1e}")
print(f"XcᵀXc + λI 的条件数：{np.linalg.cond(Xc_dup.T @ Xc_dup + 1.0 * np.eye(9)):.1e}")
print("测试 R²:", round(r2_score(y_test, X_dup_test @ w_r + b_r), 4))

两列又被平分了（各 0.426），而且这次是**方程本身唯一确定**的，换哪台电脑都一样；条件数从 1e15 以上降到约 3.5e4。测试 R² 几乎没变：λ = 1 对 15,480 个训练样本来说是很轻的惩罚。

- Ridge 是你第一次见到的**正则化**：在损失里加一项，把参数往 0 拉，换来稳定、唯一的解。下周的逻辑回归也有正则化，强度由参数 `C` 控制，方向正好相反：sklearn 把惩罚写成 ‖w‖²/(2C)，**C 越小，惩罚越强**。
- 第 14 周会看到 λ 的概率含义：它相当于事先相信"权重不会太大"——一个先验。

（选做练习：`04_exercises.ipynb` 末尾的 λ 扫描，画出 Ridge 的系数路径。）

### 预告：模型更强 = 更好？

线性模型只能画直线。给特征加上平方、立方项（多项式特征），模型就能弯曲。只取 15 个训练点试试直线和 12 次曲线：

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

X_small, y_small = X[:15], y[:15]          # 只用合成数据的前 15 个点
xs_plot = np.linspace(-2, 2, 200).reshape(-1, 1)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.scatter(X_small[:, 0], y_small, color="k", zorder=3, label="15 training points")
for degree in [1, 12]:
    poly = PolynomialFeatures(degree=degree)
    m = LinearRegression().fit(poly.fit_transform(X_small), y_small)
    ax.plot(xs_plot[:, 0], m.predict(poly.transform(xs_plot)), label=f"degree {degree}")
ax.set_ylim(-8, 12)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Degree 12 hits every point but learns the noise: overfitting")
ax.legend()
plt.show()

12 次曲线几乎穿过每个训练点——训练损失更低，但它学到的是**噪声**，在新数据上会一塌糊涂。模型更强不一定更好，怎么诊断、怎么防？这是第 5 周的主题。

### 本周小结

- 向量与矩阵：点积、`(n, d) @ (d,) → (n,)`、每一行是一个样本；shape 对不上先 print
- 导数即斜率；数值差分能验证解析梯度（sanity check）
- 四要素：X、y、模型、损失；合成数据是自带答案的阳性对照
- 梯度下降：`参数 ← 参数 - lr × 梯度`；loss 曲线是训练的心电图
- 学习率太小太慢、太大发散；标准化把峡谷变回圆碗
- 正规方程 = 令梯度为 0：(AᵀA)θ = Aᵀy 一步解出线性回归；相关的特征会留下窄峡谷，**R² 对上 ≠ 参数对上**
- XᵀX 不可逆（同一个指标记了两次）时，手写 `solve` 依电脑而报错或给出任意解，`lstsq` 给最小范数解；Ridge 加上 λI 后一定可逆——它是你见到的第一个正则化
- 真实数据有截断、有离群值：先查出处，再怀疑代码